In [17]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import broadcast

In [18]:
spark = SparkSession.builder.master("local[*]") \
    .appName("Day4") \
    .getOrCreate()           # no. of executor /driver specify garne ho mathi [] esma




In [19]:
# initializing data
path = "data" 

customers = spark.read.csv(f"{path}/customers.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{path}/orders.csv", header=True, inferSchema=True)

#orders.count() , customers.count()

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",-1)  #no auto-broadcast
spark.conf.set("spark.sql.adaptive.enabled","False")   #AQE off


In [7]:
#groupby
orders.groupBy("country") \
    .agg(F.sum("amount").alias("revenue")) \
    .orderBy(F.desc("revenue")).show()

+-------+-------------------+
|country|            revenue|
+-------+-------------------+
|     NP|1.456885329999998E7|
|     JP| 1396568.9599999997|
|     AU| 1394732.7100000002|
|     IN| 1393406.4800000004|
|     GB| 1392449.4100000015|
|     DE| 1383056.2099999997|
|     US| 1377473.9400000009|
|     BR| 1377097.1700000023|
+-------+-------------------+



In [9]:
spark.conf.set("spark.sql.shuffle.partitions",8)

In [10]:
#groupby
orders.groupBy("country") \
    .agg(F.sum("amount").alias("revenue")) \
    .orderBy(F.desc("revenue")).show()

+-------+-------------------+
|country|            revenue|
+-------+-------------------+
|     NP|1.456885329999998E7|
|     JP| 1396568.9599999997|
|     AU| 1394732.7100000002|
|     IN| 1393406.4800000004|
|     GB| 1392449.4100000015|
|     DE| 1383056.2099999997|
|     US| 1377473.9400000009|
|     BR| 1377097.1700000023|
+-------+-------------------+



In [11]:
bad = orders.join(customers, "customer_id") \
    .groupBy(customers["country"]).agg(F.sum("amount").alias("revenue"))

bad.orderBy(F.desc("revenue")).show()

+-------+------------------+
|country|           revenue|
+-------+------------------+
|     US| 3163820.560000002|
|     JP| 3098067.459999997|
|     DE|3077766.3800000018|
|     IN|3074561.1100000017|
|     GB| 3031604.960000001|
|     NP| 3007928.229999997|
|     BR| 2937445.779999996|
|     AU| 2892443.700000001|
+-------+------------------+



In [12]:
good = orders.join(broadcast(customers), "customer_id") \
    .groupBy(customers["country"]).agg(F.sum("amount").alias("revenue"))

good.orderBy(F.desc("revenue")).show()

+-------+------------------+
|country|           revenue|
+-------+------------------+
|     US|3163820.5599999935|
|     JP| 3098067.459999994|
|     DE|3077766.3800000013|
|     IN|3074561.1100000036|
|     GB|3031604.9599999967|
|     NP| 3007928.229999999|
|     BR| 2937445.779999993|
|     AU|2892443.7000000025|
+-------+------------------+



In [13]:
#skewed grodupby - AQE off
orders.groupBy("country")\
    .agg(F.countDistinct("customer_id"), F.sum("amount")).show()

+-------+---------------------------+------------------+
|country|count(DISTINCT customer_id)|       sum(amount)|
+-------+---------------------------+------------------+
|     AU|                       8138|        1394732.71|
|     IN|                       8170|        1393406.48|
|     NP|                      10000|      1.45688533E7|
|     JP|                       8199|        1396568.96|
|     GB|                       8183|1392449.4099999997|
|     DE|                       8194| 1383056.210000001|
|     US|                       8152|1377473.9399999997|
|     BR|                       8228|1377097.1700000002|
+-------+---------------------------+------------------+



In [ ]:
#fix
spark.conf.set("spark.sql.adaptive.enabled","true")
spark.conf.set("spark.sql.adaptive.skewjoin.enabled","true") 

In [15]:
orders.groupBy("country")\
    .agg(F.countDistinct("customer_id"), F.sum("amount")).show()

+-------+---------------------------+--------------------+
|country|count(DISTINCT customer_id)|         sum(amount)|
+-------+---------------------------+--------------------+
|     AU|                       8138|  1394732.7100000007|
|     IN|                       8170|          1393406.48|
|     NP|                      10000|1.4568853299999904E7|
|     JP|                       8199|  1396568.9599999993|
|     GB|                       8183|  1392449.4100000078|
|     DE|                       8194|   1383056.209999996|
|     US|                       8152|  1377473.9400000032|
|     BR|                       8228|  1377097.1699999967|
+-------+---------------------------+--------------------+



In [16]:
spark.stop()


In [20]:
base = orders.filter(F.col("amount")>0) \
    .join(broadcast(customers),"customer_id")

In [21]:
base.groupBy(customers["country"]).count().show()

+-------+-----+
|country|count|
+-------+-----+
|     NP|37098|
|     AU|36069|
|     GB|37423|
|     BR|36331|
|     DE|38150|
|     US|38830|
|     IN|37929|
|     JP|38170|
+-------+-----+



In [22]:
base.groupBy("product_id").count().show()

+----------+-----+
|product_id|count|
+----------+-----+
|       471|  579|
|       496|  567|
|       148|  613|
|       463|  592|
|       392|  589|
|       243|  589|
|        31|  608|
|       251|  634|
|       451|  618|
|        85|  602|
|       137|  553|
|        65|  614|
|       458|  612|
|       481|  584|
|        53|  575|
|       255|  615|
|       296|  587|
|       133|  588|
|       472|  587|
|        78|  619|
+----------+-----+
only showing top 20 rows


In [23]:
base.cache()
base.count() #force materialized

300000